In [2]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score
from Dataloader import Dataset, Load_Dataset_A, Load_Dataset_B, Load_Dataset_C
from LOSO_Split import Split_Dataset_A_Res, Split_Dataset_B_Res, Split_Dataset_C_Res
from Parameters import parameters
from model import fNIRS_T, fNIRS_PreT
import pandas as pd
import os

if __name__ == "__main__":
    data_path, dataset, model = parameters()

    # Load dataset, set number of Subjects
    if dataset == 'A':
        Subjects = 8
        if model == 'fNIRS-T':
            feature, label = Load_Dataset_A(data_path, model='fNIRS-T')
        elif model == 'fNIRS-PreT':
            feature, label = Load_Dataset_A(data_path, model='fNIRS-PreT')
    elif dataset == 'B':
        Subjects = 29
        feature, label = Load_Dataset_B(data_path)
    elif dataset == 'C':
        Subjects = 30
        feature, label = Load_Dataset_C(data_path)

    _, _, channels, sampling_points = feature.shape

    result_acc = []
    result_pre = []
    result_rec = []
    result_f1 = []
    result_kap = []
    for sub in range(1, Subjects+1):
        if dataset == 'A':
            X_test, y_test = Split_Dataset_A_Res(sub, feature, label, channels)
        elif dataset == 'B':
             X_test, y_test = Split_Dataset_B_Res(sub, feature, label, channels)
        elif dataset == 'C':
             X_test, y_test = Split_Dataset_C_Res(sub, feature, label, channels)

        test_set = Dataset(X_test, y_test, transform=True)
        test_loader = torch.utils.data.DataLoader(test_set, batch_size=X_test.shape[0], shuffle=False)
        # -------------------------------------------------------------------------------------------------------------------- #
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        if dataset == 'A':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
        elif dataset == 'B':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
        elif dataset == 'C':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=3, sampling_point=sampling_points, dim=128, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=3, sampling_point=sampling_points, dim=128, depth=6, heads=8, mlp_dim=64).to(device)

        # pre-trained weights
        weight_path = 'Results/' + dataset + '/LOSO/' + model + '/' + str(sub) + '/model.pt'
        net.load_state_dict(torch.load(weight_path))
        data_path = 'Results/' + dataset + '/LOSO/' + model
        log_excel = 'Results/' + dataset + '/LOSO/' + model + '/' + 'Results.xlsx'
        # -------------------------------------------------------------------------------------------------------------------- #
        net.eval()
        with torch.no_grad():
            for data in test_loader:
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                pred = outputs.argmax(dim=1, keepdim=True)

        y_label = labels.cpu()
        y_pred = pred.cpu()

        acc = accuracy_score(y_label, y_pred)

        if dataset == 'C':
            # Multi-classification using macro mode
            precision = precision_score(y_label, y_pred, average='macro')
            recall = recall_score(y_label, y_pred, average='macro')
            f1 = f1_score(y_label, y_pred, average='macro')
        else:
            precision = precision_score(y_label, y_pred)
            recall = recall_score(y_label, y_pred)
            f1 = f1_score(y_label, y_pred)
        kappa = cohen_kappa_score(y_label, y_pred)

        result_acc.append(acc)
        result_pre.append(precision)
        result_rec.append(recall)
        result_f1.append(f1)
        result_kap.append(kappa)

        print('\nAccuracy: {:.2f}%'.format(acc * 100))
        print('Precision: {:.2f}%'.format(precision * 100))
        print('Recall: {:.2f}%'.format(recall * 100))
        print("f1-score: %.2f" % (f1 * 100))
        print("kappa: %.2f" % kappa)

        # save the results
        with open(data_path + '/Metrics.txt', 'a') as f:
            f.write(f'Accuracy: {acc:.2f}\n')
            f.write(f'Precision: {precision:.2f}\n')
            f.write(f'Recall: {recall:.2f}\n')
            f.write(f'F1-Score: {f1:.2f}\n')
            f.write(f'Kappa: {kappa:.2f}\n')
        
        df = pd.DataFrame([[acc, precision, recall, f1, kappa]],
                        columns=['Accuracy','Precision','Recall','F1','Kappa'])

        if not os.path.exists(log_excel):
            df.to_excel(log_excel, index=False, engine='openpyxl')
        else:
            with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
                if 'Sheet1' in writer.sheets:
                    startrow = writer.sheets['Sheet1'].max_row
                else:
                    startrow = 0
                df.to_excel(writer, index=False, header=False, startrow=startrow)

    result_acc = np.array(result_acc)
    acc_mean, acc_std = float(np.mean(result_acc)), float(np.std(result_acc))
    result_pre = np.array(result_pre)
    pre_mean, pre_std = float(np.mean(result_pre)), float(np.std(result_pre))
    result_rec = np.array(result_rec)
    rec_mean, rec_std = float(np.mean(result_rec)), float(np.std(result_rec))
    result_f1 = np.array(result_f1)
    f1_mean, f1_std = float(np.mean(result_f1)), float(np.std(result_f1))
    result_kap = np.array(result_kap)
    kap_mean = float(np.mean(result_kap))

    print('\nacc_mean = %.2f, std = %.2f' % (acc_mean * 100, acc_std * 100))
    print('pre_mean = %.2f, std = %.2f' % (pre_mean * 100, pre_std * 100))
    print('rec_mean = %.2f, std = %.2f' % (rec_mean * 100, rec_std * 100))
    print('f1_mean = %.2f, std = %.2f' % (f1_mean * 100, f1_std * 100))
    print('kap_mean = %.2f' % kap_mean)

B fNIRS-T
1  OK:  (60, 2, 36, 200)
2  OK:  (120, 2, 36, 200)
3  OK:  (180, 2, 36, 200)
4  OK:  (240, 2, 36, 200)
5  OK:  (300, 2, 36, 200)
6  OK:  (360, 2, 36, 200)
7  OK:  (420, 2, 36, 200)
8  OK:  (480, 2, 36, 200)
9  OK:  (540, 2, 36, 200)
10  OK:  (600, 2, 36, 200)
11  OK:  (660, 2, 36, 200)
12  OK:  (720, 2, 36, 200)
13  OK:  (780, 2, 36, 200)
14  OK:  (840, 2, 36, 200)
15  OK:  (900, 2, 36, 200)
16  OK:  (960, 2, 36, 200)
17  OK:  (1020, 2, 36, 200)
18  OK:  (1080, 2, 36, 200)
19  OK:  (1140, 2, 36, 200)
20  OK:  (1200, 2, 36, 200)
21  OK:  (1260, 2, 36, 200)
22  OK:  (1320, 2, 36, 200)
23  OK:  (1380, 2, 36, 200)
24  OK:  (1440, 2, 36, 200)
25  OK:  (1500, 2, 36, 200)
26  OK:  (1560, 2, 36, 200)
27  OK:  (1620, 2, 36, 200)
28  OK:  (1680, 2, 36, 200)
29  OK:  (1740, 2, 36, 200)
feature  (1740, 2, 36, 200)
label  (1740,)
torch.Size([60, 2, 36, 200])
torch.Size([60])

Accuracy: 71.67%
Precision: 88.24%
Recall: 50.00%
f1-score: 63.83
kappa: 0.43
torch.Size([60, 2, 36, 200])
torch.S